# Financial Data with Yahoo Finance

This notebook keeps the original **three-part teaching architecture** of the finance module, but replaces the China-market AKShare example with a more internationally relevant case based on **Yahoo Finance** data through the `yfinance` package.

This version is designed for a more global teaching audience and is especially suitable if you want to present:
- cross-market investment ideas rather than one domestic exchange,
- a compact institutional watchlist,
- reproducible code that students can run in **GitHub + Colab**.

**Practical note:** Online finance APIs can change over time. Before teaching or recording a video, rerun the notebook once and check for missing values or temporary download issues.


In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT


In [ ]:
MODULE_OUTPUT_DIR = OUTPUT_DIR / "06_financial_data"
MODULE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODULE_OUTPUT_DIR


**Optional setup**

In [ ]:
%pip install -q yfinance ipywidgets seaborn scipy

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import scipy.optimize as sco

warnings.filterwarnings("ignore")

TRADING_DAYS = 252
RISK_FREE_RATE = 0.02
START_DATE = "2022-01-01"
END_DATE = "2025-01-01"

def normalize_datetime_column(df: pd.DataFrame, col: str = "date") -> pd.DataFrame:
    df = df.copy()
    df[col] = pd.to_datetime(df[col])
    try:
        df[col] = df[col].dt.tz_localize(None)
    except (TypeError, AttributeError):
        pass
    return df

def get_price_block(downloaded: pd.DataFrame, field: str) -> pd.DataFrame:
    """Extract a field such as 'Open' or 'Close' from a multi-ticker yfinance download."""
    if isinstance(downloaded.columns, pd.MultiIndex):
        block = downloaded.xs(field, axis=1, level=0)
    else:
        block = downloaded[[field]].copy()
    block.columns = [str(c) for c in block.columns]
    return block

def wide_to_long(price_df: pd.DataFrame, value_name: str) -> pd.DataFrame:
    out = price_df.stack(dropna=False).rename(value_name).reset_index()
    out.columns = ["date", "ticker", value_name]
    return out


# Part 1. Obtain market data and compute basic statistics

# Q1. How can we check ticker symbols and names for an institutional watchlist?

In international practice, analysts often begin with a **curated investment universe** rather than scraping one domestic exchange.

The example below uses four liquid, globally familiar ETFs:
- `SPY`: U.S. large-cap equities
- `VEA`: developed markets equities outside the U.S.
- `EEM`: emerging markets equities
- `AGG`: investment-grade U.S. bonds

This makes the later portfolio-allocation section more suitable for an international audience.


In [ ]:
institutional_universe = pd.DataFrame(
    {
        "ticker": ["SPY", "VEA", "EEM", "AGG"],
        "instrument_name": [
            "SPDR S&P 500 ETF Trust",
            "Vanguard FTSE Developed Markets ETF",
            "iShares MSCI Emerging Markets ETF",
            "iShares Core U.S. Aggregate Bond ETF",
        ],
        "asset_class": ["Equity", "Equity", "Equity", "Bond"],
        "market_focus": [
            "United States",
            "Developed Markets ex U.S.",
            "Emerging Markets",
            "U.S. Investment-Grade Bonds",
        ],
        "teaching_role": [
            "Core U.S. equity proxy",
            "Developed-market diversification",
            "Emerging-market diversification",
            "Defensive fixed-income allocation",
        ],
    }
)

ticker_list = institutional_universe["ticker"].tolist()
instrument_name_map = dict(
    zip(institutional_universe["ticker"], institutional_universe["instrument_name"])
)

institutional_universe


In [ ]:
# Recent price snapshot for the selected universe.
recent_download = yf.download(
    tickers=ticker_list,
    period="5d",
    interval="1d",
    auto_adjust=True,
    progress=False,
    threads=False,
    group_by="column",
)

recent_close = get_price_block(recent_download, "Close")
latest_snapshot = recent_close.ffill().iloc[-1].rename("latest_close").to_frame().reset_index()
latest_snapshot.columns = ["ticker", "latest_close"]
latest_snapshot = latest_snapshot.merge(institutional_universe, on="ticker", how="left")

latest_snapshot


# Q2. How can we obtain market data?

## Q2.1. Single-instrument example: SPDR S&P 500 ETF Trust (`SPY`)

In [ ]:
spy = yf.Ticker("SPY").history(
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
)

spy = spy.reset_index()
spy.columns = [str(col).lower().replace(" ", "_") for col in spy.columns]
spy = normalize_datetime_column(spy, "date")
spy["daily_return"] = spy["close"].pct_change()

spy.head()


## Q2.2. Multi-instrument example

In [ ]:
multi_download = yf.download(
    tickers=ticker_list,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False,
    threads=False,
    group_by="column",
)

close_prices = get_price_block(multi_download, "Close")
open_prices = get_price_block(multi_download, "Open")

close_long = wide_to_long(close_prices, "close")
open_long = wide_to_long(open_prices, "open")

market_df = (
    close_long
    .merge(open_long, on=["date", "ticker"], how="left")
    .dropna(subset=["close"])
)

market_df = normalize_datetime_column(market_df, "date")
market_df["daily_return"] = market_df.groupby("ticker")["close"].pct_change()
market_df["instrument_name"] = market_df["ticker"].map(instrument_name_map)

market_df.head()


# Q3. How can we calculate annualized return, annualized volatility, and the Sharpe ratio?

Assume **252 trading days per year**, which is a common convention in international equity-market analysis.

- Annualized return = mean daily return × 252
- Annualized volatility = standard deviation of daily return × √252
- Sharpe ratio = (annualized return − risk-free rate) / annualized volatility

This notebook uses an **assumed** annualized risk-free rate of `0.02` (2.0%) for demonstration.


In [ ]:
spy[["date", "close", "daily_return"]].head()

In [ ]:
spy_return = spy["daily_return"].mean() * TRADING_DAYS
spy_return


In [ ]:
spy_volatility = spy["daily_return"].std() * np.sqrt(TRADING_DAYS)
spy_volatility


In [ ]:
spy_sharpe = (spy_return - RISK_FREE_RATE) / spy_volatility
spy_sharpe


# Q4. Grouped descriptive statistics

In [ ]:
summary_table = (
    market_df.groupby("instrument_name")[["daily_return", "close"]]
    .describe()
    .round(4)
    .T
)

summary_table


# Part 2. Visualize market data

# Q5. How can we create basic visualizations for market data?

## Q5.1. Opening and closing prices for SPY

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(spy["date"], spy["open"], linestyle="-.", linewidth=1, marker=".", alpha=0.5)
plt.plot(spy["date"], spy["close"])
plt.xlabel("Date")
plt.ylabel("Price")
plt.title("SPY: Open vs. Close")
plt.legend(["Opening price", "Closing price"], loc="best")
plt.tight_layout()
plt.savefig(MODULE_OUTPUT_DIR / "yfinance_open_close_lineplot.png", bbox_inches="tight")
plt.show()


In [ ]:
%pip install -q seaborn

## Q5.2. Multi-instrument line plot

In [ ]:
plt.figure(figsize=(12, 5))
sns.lineplot(data=market_df, x="date", y="daily_return", hue="instrument_name")
plt.title("Daily Return Series")
plt.xlabel("Date")
plt.ylabel("Daily return")
plt.tight_layout()
plt.savefig(MODULE_OUTPUT_DIR / "yfinance_multi_asset_daily_return_lineplot.png", bbox_inches="tight")
plt.show()


## Q5.3. Histograms for multiple instruments

In [ ]:
g = sns.FacetGrid(market_df.dropna(subset=["daily_return"]), col="instrument_name", col_wrap=2, sharex=False, sharey=False)
g.map(sns.histplot, "daily_return", bins=20)
g.set_titles("{col_name}")
g.fig.tight_layout()


## Q5.4. Cumulative return plot

In [ ]:
market_df = market_df.sort_values(["ticker", "date"]).copy()

# For performance charts, compounded returns are more appropriate than a simple cumulative sum.
market_df["cumulative_return"] = (
    market_df.groupby("ticker")["daily_return"]
    .transform(lambda s: (1 + s.fillna(0)).cumprod() - 1)
)

plt.figure(figsize=(12, 5))
sns.lineplot(data=market_df, x="date", y="cumulative_return", hue="instrument_name")
plt.title("Compounded Cumulative Return")
plt.xlabel("Date")
plt.ylabel("Cumulative return")
plt.tight_layout()
plt.savefig(MODULE_OUTPUT_DIR / "yfinance_cumulative_return_lineplot.png", bbox_inches="tight")
plt.show()


# Part 3. Markowitz mean-variance portfolio model

This section uses Monte Carlo simulation and constrained optimization to explore:
- random portfolios,
- the maximum-Sharpe portfolio,
- the minimum-volatility portfolio,
- and the efficient frontier.

To keep the interpretation clean, we use a **long-only** portfolio (`0 ≤ weight ≤ 1`) with the full-investment constraint (`sum of weights = 1`).


In [ ]:
returns = (
    market_df[["ticker", "daily_return", "date"]]
    .pivot_table(values="daily_return", index="date", columns="ticker")
    .dropna(how="all")
    .dropna()
)

returns.head()


# Q6. How can we generate many random portfolios with Monte Carlo simulation?

In [ ]:
noa = len(ticker_list)

portfolio_returns = []
portfolio_volatility = []
portfolio_sharpe = []
portfolio_weights = []

for _ in range(5000):
    weights = np.random.random(noa)
    weights /= np.sum(weights)

    port_return = np.sum(returns.mean() * TRADING_DAYS * weights)
    port_volatility = np.sqrt(np.dot(weights.T, np.dot(returns.cov() * TRADING_DAYS, weights)))
    sharpe_ratio = (port_return - RISK_FREE_RATE) / port_volatility

    portfolio_returns.append(port_return)
    portfolio_volatility.append(port_volatility)
    portfolio_sharpe.append(sharpe_ratio)
    portfolio_weights.append(weights)

portfolio_returns = np.array(portfolio_returns)
portfolio_volatility = np.array(portfolio_volatility)
portfolio_sharpe = np.array(portfolio_sharpe)


In [ ]:
plt.figure(figsize=(8, 4))
plt.scatter(portfolio_volatility, portfolio_returns, c=portfolio_sharpe, marker="o")
plt.xlabel("Expected volatility")
plt.ylabel("Expected return")
plt.title("Monte Carlo Portfolio Simulation")
plt.grid(True)
plt.colorbar(label="Sharpe ratio")
plt.tight_layout()
plt.savefig(MODULE_OUTPUT_DIR / "yfinance_monte_carlo_portfolios.png", bbox_inches="tight")
plt.show()


# Q7. How can we find the portfolio with the largest Sharpe ratio?

In [ ]:
max_sharpe_idx = np.argmax(portfolio_sharpe)
max_sharpe_return = portfolio_returns[max_sharpe_idx]
max_sharpe_volatility = portfolio_volatility[max_sharpe_idx]
max_sharpe_ratio = portfolio_sharpe[max_sharpe_idx]

print(f"Largest Sharpe ratio: {max_sharpe_ratio:.4f}")
print(f"Expected return: {max_sharpe_return:.4f}")
print(f"Expected volatility: {max_sharpe_volatility:.4f}")


In [ ]:
def statistics(weights):
    weights = np.array(weights)
    port_return = np.sum(returns.mean() * TRADING_DAYS * weights)
    port_volatility = np.sqrt(np.dot(weights.T, np.dot(returns.cov() * TRADING_DAYS, weights)))
    sharpe = (port_return - RISK_FREE_RATE) / port_volatility
    return np.array([port_return, port_volatility, sharpe])

def negative_sharpe(weights):
    return -statistics(weights)[2]

bounds = ((0, 1),) * noa
constraints = {"type": "eq", "fun": lambda x: np.sum(x) - 1}

opt_sharpe = sco.minimize(
    negative_sharpe,
    noa * [1.0 / noa],
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
)

max_sharpe_weights = pd.DataFrame(
    {
        "ticker": ticker_list,
        "instrument_name": [instrument_name_map[t] for t in ticker_list],
        "weight": opt_sharpe.x,
    }
).sort_values("weight", ascending=False)

max_sharpe_weights


# Q8. How can we find the portfolio that minimizes volatility?

In [ ]:
def portfolio_vol(weights):
    return statistics(weights)[1]

opt_min_vol = sco.minimize(
    portfolio_vol,
    noa * [1.0 / noa],
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
)

min_vol_weights = pd.DataFrame(
    {
        "ticker": ticker_list,
        "instrument_name": [instrument_name_map[t] for t in ticker_list],
        "weight": opt_min_vol.x,
    }
).sort_values("weight", ascending=False)

print(f"Minimum volatility: {opt_min_vol.fun:.4f}")
min_vol_weights


# Q9. How can we draw the efficient frontier?

In [ ]:
target_returns = np.linspace(
    (returns.mean() * TRADING_DAYS).min(),
    (returns.mean() * TRADING_DAYS).max(),
    100,
)

target_volatility = []

for target in target_returns:
    target_constraints = (
        {"type": "eq", "fun": lambda x, target=target: statistics(x)[0] - target},
        {"type": "eq", "fun": lambda x: np.sum(x) - 1},
    )

    result = sco.minimize(
        portfolio_vol,
        noa * [1.0 / noa],
        method="SLSQP",
        bounds=bounds,
        constraints=target_constraints,
    )

    target_volatility.append(result["fun"])

target_volatility = np.array(target_volatility)


In [ ]:
plt.figure(figsize=(8, 4))
plt.scatter(portfolio_volatility, portfolio_returns, c=portfolio_sharpe, marker="o", alpha=0.7)
plt.scatter(target_volatility, target_returns, c=target_returns / target_volatility, cmap="autumn", marker="x", s=8)
plt.plot(statistics(opt_sharpe["x"])[1], statistics(opt_sharpe["x"])[0], "r*", markersize=15)
plt.plot(statistics(opt_min_vol["x"])[1], statistics(opt_min_vol["x"])[0], "y*", markersize=15)
plt.xlabel("Expected volatility")
plt.ylabel("Expected return")
plt.title("Efficient Frontier")
plt.grid(True)
plt.colorbar(label="Return / volatility")
plt.tight_layout()
plt.savefig(MODULE_OUTPUT_DIR / "yfinance_efficient_frontier.png", bbox_inches="tight")
plt.show()


## Teaching notes

1. This case keeps the original lecture logic, but the investment universe is now more internationally recognizable.
2. The cumulative-return chart uses **compounded** returns, which is more appropriate than adding daily returns directly.
3. In the optimization section, the Sharpe-ratio objective explicitly subtracts the chosen risk-free rate.
4. You can easily adapt the notebook later for:
   - sector ETFs,
   - country ETFs,
   - global equity indices,
   - or a portfolio that mixes equities, bonds, and commodities.
